# Rubrics and pointwise scoring [Step 07.01 - Building a judge you can defend]

> **MLCourse - Agentic AI - Agent Patterns**

You have already *used* an LLM judge twice in this course - in
[`../../03_rag_advanced/10_rag_evaluation`](../../03_rag_advanced/10_rag_evaluation)
and in [`../06_multi_agent_debate`](../06_multi_agent_debate). Neither time did we
ask the obvious question: **is the judge any good?**

An LLM judge is a model asked to score or compare outputs instead of producing
them. It is the only practical way to evaluate open-ended text at volume - humans
are too slow and too expensive, and exact-match metrics cannot read.

It is also a measuring instrument, and **an uncalibrated instrument is not a
measurement, it is a number.** This module calibrates one.

### The two shapes

```
POINTWISE                              PAIRWISE
 one output at a time                   two outputs, head to head
 "score this 1-5 on correctness"        "which is more correct, A or B?"
   + absolute, comparable over time       + much easier for a model to do well
   + cheap: N calls for N outputs         + no scale drift
   - scale drift; clustering at 3-4       - O(N^2) comparisons for a ranking
   - very sensitive to the rubric         - POSITION BIAS (notebook 02)
```

### What you'll learn

- What a rubric has to contain before a judge can use it: **anchors**, not adjectives.
- Reference-free vs reference-based judging, and why the second is far more reliable.
- Pointwise scoring in practice, and the score-clustering problem you will hit.

### Key takeaways

- "Rate this answer 1-5 for quality" is not a rubric. It is a vibe.
- Every point on the scale needs a **description you could defend in an argument**.
- The judge must be forced into a machine-readable verdict, parsed by a regex.
  Never use a second LLM to read the first one's output.
- Give the judge a **reference answer** whenever you have one. It converts an
  opinion into a comparison.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 2.0
print("PACE =", PACE)

PACE = 2.0


### The evaluation pairs


In [ ]:
# Five questions, each with a GOOD answer and a WORSE answer. "Worse" is worse for
# a stated, checkable reason - not just shorter or blander - so we know the right
# verdict without asking anyone.
#
# Building the pairs by hand like this is the only way to measure a judge: you need
# ground truth ABOUT THE JUDGE, which means you must know the answer already.

PAIRS = [
    dict(
        id="photosynthesis",
        q="In one or two sentences, what does photosynthesis produce?",
        good="Photosynthesis produces glucose and oxygen, using carbon dioxide, water and light energy.",
        bad="Photosynthesis produces carbon dioxide and water, which the plant then releases into the air.",
        why_bad="reverses the reactants and the products - factually wrong",
    ),
    dict(
        id="http_status",
        q="What does HTTP status code 404 mean?",
        good="404 means the server understood the request but could not find the requested resource.",
        bad="404 means the server is temporarily overloaded and the client should retry later.",
        why_bad="describes 503, not 404",
    ),
    dict(
        id="python_list",
        q="What is the time complexity of appending to a Python list?",
        good="Amortised O(1): appends are constant time on average, with occasional O(n) reallocations.",
        bad="O(n), because the list has to be copied every time an element is added.",
        why_bad="wrong complexity - ignores the amortised growth strategy",
    ),
    dict(
        id="vaccine",
        q="Briefly, how do mRNA vaccines work?",
        good=("They deliver mRNA instructing your cells to make a harmless viral protein, "
              "which the immune system then learns to recognise."),
        bad=("They inject a weakened live virus that reproduces slowly so the immune system "
             "can practise fighting it."),
        why_bad="describes a live attenuated vaccine, not an mRNA vaccine",
    ),
    dict(
        id="git_rebase",
        q="What does `git rebase` do, in one sentence?",
        good="It replays your commits on top of another base commit, rewriting their history.",
        bad="It merges two branches together and creates a merge commit recording both parents.",
        why_bad="describes merge, which is the thing rebase is contrasted with",
    ),
]

print("%d pairs" % len(PAIRS))
for p in PAIRS:
    print("  %-16s bad answer: %s" % (p["id"], p["why_bad"]))


### 1. The bad rubric

Start with what most people write first, so the failure is concrete.

In [4]:
import re

vague_judge = make_llm(temperature=0.0, max_tokens=180)

VAGUE = ("Rate the following answer from 1 to 5 for quality. "
         "Reply with a final line 'SCORE: <n>'.")

SCORE_RE = re.compile(r"SCORE:\s*([1-5])")

vague_scores = {}
for p in PAIRS:
    for label in ("good", "bad"):
        m = safe_invoke(vague_judge, [("system", VAGUE),
                                      ("user", "QUESTION: %s\n\nANSWER: %s" % (p["q"], p[label]))])
        s = SCORE_RE.search(m.content)
        vague_scores[(p["id"], label)] = int(s.group(1)) if s else None
        print("%-16s %-5s -> %s" % (p["id"], label, vague_scores[(p["id"], label)]))

photosynthesis   good  -> 5


photosynthesis   bad   -> 1


http_status      good  -> 5


http_status      bad   -> 1


python_list      good  -> 5


python_list      bad   -> None


vaccine          good  -> 5


vaccine          bad   -> 1


git_rebase       good  -> 5


git_rebase       bad   -> 1


In [5]:
def mean(xs):
    """Mean over the scores that PARSED. Never silently substitute a default:
    an unparsed verdict is missing data, not a 3."""
    vals = [x for x in xs if x is not None]
    return (sum(vals) / len(vals)) if vals else float("nan")


def n_missing(xs):
    return sum(1 for x in xs if x is None)


goods = [vague_scores[(p["id"], "good")] for p in PAIRS]
bads = [vague_scores[(p["id"], "bad")] for p in PAIRS]
sep = sum(1 for g, b in zip(goods, bads) if g is not None and b is not None and g > b)

print("VAGUE RUBRIC")
print("  good answers : %s   mean %.2f" % (goods, mean(goods)))
print("  bad answers  : %s   mean %.2f" % (bads, mean(bads)))
print("  separated correctly (good > bad): %d/%d" % (sep, len(PAIRS)))
print("  distinct scores used: %s" % sorted(set(x for x in goods + bads if x is not None)))
print("  UNPARSED verdicts: %d (these are missing data, not middling scores)"
      % (n_missing(goods) + n_missing(bads)))

VAGUE RUBRIC
  good answers : [5, 5, 5, 5, 5]   mean 5.00
  bad answers  : [1, 1, None, 1, 1]   mean 1.00
  separated correctly (good > bad): 4/5
  distinct scores used: [1, 5]
  UNPARSED verdicts: 1 (these are missing data, not middling scores)


Two things to look for in that output, both of which are the standard result:

- **Score clustering.** The judge uses two or three values out of five. A 1-5 scale
  that in practice is a 3-5 scale has about 1.5 bits of resolution.
- **Grade inflation.** Factually wrong answers still score 3+, because "quality"
  invites the judge to reward fluency, and the bad answers here are fluent.

The bad answers above are not badly *written*. They are wrong. A rubric that does
not say what is being measured will measure writing.

### 2. A rubric with anchors

A usable rubric has three properties:

1. **One dimension per judgement.** Correctness and helpfulness are different
   questions; asking for one "quality" number averages them invisibly.
2. **An anchor for every scale point** - a sentence describing what earns that
   score, specific enough that two people would agree.
3. **A required output format** a regex can read.

Optionally but powerfully: a **reference answer**. Judging "is this correct" is
hard; judging "does this agree with that" is much easier.

In [6]:
ANCHORED = """You are grading the FACTUAL CORRECTNESS of an answer. Nothing else:
ignore style, length, tone, formatting and confidence.

Use exactly this scale:
5 - Every factual claim is correct, and the answer addresses the question asked.
4 - Correct overall; a minor imprecision that would not mislead a reader.
3 - Partially correct; a mixture of correct and incorrect claims.
2 - Mostly incorrect; a reader would come away with a wrong understanding.
1 - The central claim is wrong, or the answer describes something else entirely.

First quote the single claim you based your score on. Then output a final line of
exactly the form 'SCORE: <n>'."""

anchored_judge = make_llm(temperature=0.0, max_tokens=200)

anchored_scores = {}
for p in PAIRS:
    for label in ("good", "bad"):
        m = safe_invoke(anchored_judge, [("system", ANCHORED),
                                         ("user", "QUESTION: %s\n\nANSWER: %s" % (p["q"], p[label]))])
        s = SCORE_RE.search(m.content)
        anchored_scores[(p["id"], label)] = int(s.group(1)) if s else None
    print("%-16s good=%s bad=%s   (bad is wrong because: %s)"
          % (p["id"], anchored_scores[(p["id"], "good")],
             anchored_scores[(p["id"], "bad")], p["why_bad"]))

photosynthesis   good=5 bad=1   (bad is wrong because: reverses the reactants and the products - factually wrong)


http_status      good=5 bad=1   (bad is wrong because: describes 503, not 404)


python_list      good=5 bad=2   (bad is wrong because: wrong complexity - ignores the amortised growth strategy)


vaccine          good=5 bad=1   (bad is wrong because: describes a live attenuated vaccine, not an mRNA vaccine)


git_rebase       good=5 bad=1   (bad is wrong because: describes merge, which is the thing rebase is contrasted with)


In [7]:
g2 = [anchored_scores[(p["id"], "good")] for p in PAIRS]
b2 = [anchored_scores[(p["id"], "bad")] for p in PAIRS]
sep2 = sum(1 for g, b in zip(g2, b2) if g is not None and b is not None and g > b)

print("=" * 64)
print("%-22s %-24s %-16s" % ("rubric", "mean good / mean bad", "separated"))
print("-" * 64)
print("%-22s %-24s %d/%d" % ("vague ('quality 1-5')",
                             "%.2f / %.2f" % (mean(goods), mean(bads)), sep, len(PAIRS)))
print("%-22s %-24s %d/%d" % ("anchored (correctness)",
                             "%.2f / %.2f" % (mean(g2), mean(b2)), sep2, len(PAIRS)))
print("-" * 64)
print("gap (good - bad):  vague %.2f   anchored %.2f"
      % (mean(goods) - mean(bads), mean(g2) - mean(b2)))
print("=" * 64)

rubric                 mean good / mean bad     separated       
----------------------------------------------------------------
vague ('quality 1-5')  5.00 / 1.00              4/5
anchored (correctness) 5.00 / 1.20              5/5
----------------------------------------------------------------
gap (good - bad):  vague 4.00   anchored 3.80


Report whatever gap you measured. The number that matters is not the mean score -
it is the **separation**: how far apart the judge puts things you know are
different. A judge that scores everything 4.0 +/- 0.2 cannot rank your system
versions no matter how eloquent its justifications are.

### 3. Reference-based judging

If you have a known-good answer, hand it over. The judge's task changes from
"evaluate this claim about the world" (which needs knowledge) to "does this agree
with that" (which needs reading). This is strictly easier and measurably more
reliable.

In [8]:
REFERENCE = """You are grading an answer against a REFERENCE answer known to be correct.

5 - Fully consistent with the reference; no contradictions.
4 - Consistent, but omits something the reference considers important.
3 - Partly consistent; contains at least one claim the reference does not support.
2 - Contradicts the reference on a substantive point.
1 - Contradicts the reference's central claim.

Quote the contradiction if there is one. Then a final line 'SCORE: <n>'."""

ref_judge = make_llm(temperature=0.0, max_tokens=200)
ref_scores = {}
for p in PAIRS:
    for label in ("good", "bad"):
        m = safe_invoke(ref_judge, [
            ("system", REFERENCE),
            ("user", "QUESTION: %s\n\nREFERENCE ANSWER: %s\n\nANSWER TO GRADE: %s"
                     % (p["q"], p["good"], p[label]))])
        s = SCORE_RE.search(m.content)
        ref_scores[(p["id"], label)] = int(s.group(1)) if s else None

g3 = [ref_scores[(p["id"], "good")] for p in PAIRS]
b3 = [ref_scores[(p["id"], "bad")] for p in PAIRS]
print("reference-based: mean good %.2f, mean bad %.2f, gap %.2f, separated %d/%d"
      % (mean(g3), mean(b3), mean(g3) - mean(b3),
         sum(1 for g, b in zip(g3, b3) if g is not None and b is not None and g > b),
         len(PAIRS)))
print()
print("NOTE: the 'good' answer IS the reference here, so its score of ~5 is not")
print("evidence of anything - it is a sanity check that the judge can read. The")
print("informative number is the BAD column and the size of the gap.")

reference-based: mean good 5.00, mean bad 1.00, gap 4.00, separated 5/5

NOTE: the 'good' answer IS the reference here, so its score of ~5 is not
evidence of anything - it is a sanity check that the judge can read. The
informative number is the BAD column and the size of the gap.


In [9]:
print("all three rubrics, side by side (good / bad)")
print("-" * 70)
print("%-16s %-14s %-14s %-14s" % ("item", "vague", "anchored", "reference"))
print("-" * 70)
for p in PAIRS:
    i = p["id"]
    print("%-16s %-14s %-14s %-14s"
          % (i,
             "%s / %s" % (vague_scores[(i, "good")], vague_scores[(i, "bad")]),
             "%s / %s" % (anchored_scores[(i, "good")], anchored_scores[(i, "bad")]),
             "%s / %s" % (ref_scores[(i, "good")], ref_scores[(i, "bad")])))
print("-" * 70)

all three rubrics, side by side (good / bad)
----------------------------------------------------------------------
item             vague          anchored       reference     
----------------------------------------------------------------------
photosynthesis   5 / 1          5 / 1          5 / 1         
http_status      5 / 1          5 / 1          5 / 1         
python_list      5 / None       5 / 2          5 / 1         
vaccine          5 / 1          5 / 1          5 / 1         
git_rebase       5 / 1          5 / 1          5 / 1         
----------------------------------------------------------------------


### Pitfalls

- **Multi-dimensional rubrics scored as one number.** Score each dimension
  separately, then combine with weights *you* choose - not weights the model
  invented silently.
- **Letting the judge answer the question itself.** If the rubric lets it reason
  about the topic, it becomes a competing answerer with the last word.
- **max_tokens too small.** A truncated reply loses the `SCORE:` line and your
  parse fails. Watch for `None` in your results - never silently treat it as a 3.
- **Temperature above 0.** A judge should be reproducible. Run it at 0 and re-run
  it periodically to confirm it still is.
- **Changing the rubric mid-experiment.** Scores from two rubrics are not
  comparable. Version your rubric alongside your code.

### Next

Notebook 02 takes the pairwise judge and measures the defect that invalidates more
LLM evaluations than any other: **position bias**.